# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and object
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets and their field schemas by `@id`.

**List all available record sets and their field `@id`s:**

In [ ]:
# List all record sets, their @id, and contained fields
print("Available record sets and their fields:\n")
record_sets = []
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    field_ids = [f"{field.id}" for field in rs.fields]
    print(f"  Field @ids: {field_ids}")
    record_sets.append(rs.id)

# For demonstration, print the first 1-2 records from each record set
print("\nSample records from each RecordSet:")
for rs_id in record_sets:
    print(f"\n-- Records from RecordSet: {rs_id} --")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Extract data from each record set into DataFrames by their @id
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Explore one DataFrame in detail
if dataframes:
    # Select the first record set for exploration
    main_record_set_id = record_sets[0]
    print(f"Columns in DataFrame (from RecordSet {main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id`s. Example: filtering on a numeric field such as peptide counts or abundance, normalizing, and grouping by a categorical field.

In [ ]:
import numpy as np

# Choose meaningful field @ids from the DataFrame columns
if dataframes:
    main_df = dataframes[main_record_set_id]
    print(f"First few columns available: {main_df.columns[:5].tolist()}")

    # Let's find a numeric field (e.g. cr:peptide_count if it exists, otherwise first numeric field)
    numeric_field = None
    potential_numeric_fields = [col for col in main_df.columns 
                              if any(x in col.lower() for x in ['abundance', 'count', 'number', 'mw', 'mass', 'value', 'area', 'coverage', 'score'])]
    for col in potential_numeric_fields:
        if np.issubdtype(main_df[col].dtype, np.number):
            numeric_field = col
            break
    if not numeric_field:
        # Try to convert object columns that may be numeric
        for col in potential_numeric_fields:
            try:
                main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
                if np.issubdtype(main_df[col].dtype, np.number):
                    numeric_field = col
                    break
            except Exception:
                pass

    if numeric_field:
        print(f"Using numeric field for EDA: {numeric_field}")
        threshold = main_df[numeric_field].median()
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (median): {len(filtered_df)} records")
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (e.g. cr:accession)
        group_field = None
        for col in main_df.columns:
            if col != numeric_field and (main_df[col].dtype == object or main_df[col].dtype.name == 'category'):
                group_field = col
                # Prefer something like accession, description, or sample_id
                if any(x in col.lower() for x in ['sample', 'accession', 'description', 'group', 'category', 'type', 'cell']):
                    break
        print(f"Grouping by field: {group_field}")
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No main DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields (using field `@id` columns in the current DataFrame).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping field available, boxplot
    if group_field and main_df[group_field].nunique() < 20:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print("No data or numeric field available for plotting.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, explore, and analyze a FAIR-compliant biological dataset. We referenced all record sets and fields using their Croissant `@id` fields.

Key steps included:
- Listing all available record sets and their field `@id`s
- Loading all records as DataFrames by record set `@id`
- Demonstrating filtering, normalization, and grouping by field `@id`
- Visualizing numeric field distributions

**This notebook provides a reproducible template for FAIR data exploration with Croissant-powered datasets.**